# HOMO-LUMO and Vibrational Frequency (NWChem)

Calculate frontier-orbital energies (HOMO, LUMO, and the HOMO-LUMO gap) and vibrational-frequency-derived thermochemistry (zero-point energy and thermal corrections) for a molecule using NWChem on the Mat3ra platform.

Choose which calculations to run with the `CALCULATE_HOMO_LUMO` and `CALCULATE_FREQUENCY` toggles below. The frequency calculation returns thermochemical quantities (zero-point energy and thermal corrections); per-mode frequencies and IR spectra are not produced.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and job

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "H2O"  # Molecule to load from the uploads folder (e.g. "H2O", "CH4", "NH3", "O2", "N2")

# 4. Workflow parameters
APPLICATION_NAME = "nwchem"

# Choose which calculations to run
CALCULATE_HOMO_LUMO = True  # HOMO energy, LUMO energy, and HOMO-LUMO gap
CALCULATE_FREQUENCY = True  # zero-point energy and thermal corrections
assert CALCULATE_HOMO_LUMO or CALCULATE_FREQUENCY, "Enable at least one of CALCULATE_HOMO_LUMO / CALCULATE_FREQUENCY"

calculation_labels = []
if CALCULATE_HOMO_LUMO:
    calculation_labels.append("HOMO-LUMO")
if CALCULATE_FREQUENCY:
    calculation_labels.append("Frequency")
MY_WORKFLOW_NAME = " + ".join(calculation_labels) + " (nwchem)"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds


## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API Client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load molecule from local file

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME)

visualize(material)


### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Create workflow and set its parameters
### 4.1. Get list of applications and select one

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Create workflow from standard workflows and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode import Workflow, Subworkflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

nwchem_workflows = WorkflowStandata.filter_by_application(app.name)

base_search_term = "total_energy.json" if CALCULATE_HOMO_LUMO else "frequency.json"
workflow = Workflow.create(nwchem_workflows.get_by_name_first_match(base_search_term))

if CALCULATE_HOMO_LUMO and CALCULATE_FREQUENCY:
    frequency_config = nwchem_workflows.get_by_name_first_match("frequency.json")
    workflow.add_subworkflow(Subworkflow(**frequency_config["subworkflows"][0]))

workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)


### 4.4. Save workflow to collection

In [ ]:
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

workflow_id_or_dict = None

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the job


In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job with material and workflow configuration
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON

print(f"Material: {saved_material.id}")
print(f"Workflow: {saved_workflow.id}")
print(f"Project: {project_id}")

formula = saved_material.formula or material.formula or MATERIAL_NAME
job_name = MY_WORKFLOW_NAME + " " + formula + " " + timestamp
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict()
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve results

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

properties_data = client.properties.get_for_job(job_id)
visualize_properties(properties_data)


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

if CALCULATE_HOMO_LUMO:
    homo_data = client.properties.get_for_job(job_id, property_name="homo_energy")
    lumo_data = client.properties.get_for_job(job_id, property_name="lumo_energy")
    visualize_properties(homo_data, title="HOMO Energy")
    visualize_properties(lumo_data, title="LUMO Energy")
    homo_lumo_gap = lumo_data[0]["value"] - homo_data[0]["value"]
    print(f"HOMO-LUMO gap: {homo_lumo_gap:.4f} eV")


In [ ]:
if CALCULATE_FREQUENCY:
    zero_point_energy_data = client.properties.get_for_job(job_id, property_name="zero_point_energy")
    thermal_correction_to_energy_data = client.properties.get_for_job(job_id, property_name="thermal_correction_to_energy")
    thermal_correction_to_enthalpy_data = client.properties.get_for_job(job_id, property_name="thermal_correction_to_enthalpy")
    visualize_properties(zero_point_energy_data, title="Zero Point Energy")
    visualize_properties(thermal_correction_to_energy_data, title="Thermal Correction to Energy")
    visualize_properties(thermal_correction_to_enthalpy_data, title="Thermal Correction to Enthalpy")
